# Cross-Lingual POS Tagging: SynPOS vs MRL-POS

**Typological hypothesis:** Morphological inductive bias (MRL-POS) helps agglutinative
languages; syntactic inductive bias (SynPOS) helps fusional languages.

| Language | Treebank | Script | Typology | Train size | Expected winner |
|----------|----------|--------|----------|------------|-----------------|
| Hindi | UD HDTB | Devanagari | Fusional | 13,304 | SynPOS |
| Marathi | UD UFAL | Devanagari | Fusional | 3,131 | SynPOS |
| Urdu | UD UDTB | Nastaliq | Fusional | 4,043 | SynPOS |
| Turkish | UD IMST | Latin | Agglutinative | 3,664 | MRL-POS |

**Controlled experiment:** Same encoder (XLM-R), same co-attention, same gating, same CRF,
same hyperparameters. Only the feature branch and language change.

**Usage:** Set `LANGUAGE` and `MODEL_TYPE` in the config cell, then Run All.

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers pytorch-crf seqeval regex conllu

In [ ]:
# Cell 2: Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import XLMRobertaModel, XLMRobertaTokenizer
from torchcrf import CRF
from collections import Counter
from sklearn.metrics import f1_score as sklearn_f1, accuracy_score, classification_report
from transformers import get_cosine_schedule_with_warmup
import numpy as np
import os
import regex
import json
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Cell 3: Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/mrl_pos_checkpoints/crosslingual'
    RESULTS_DIR = '/content/drive/MyDrive/mrl_pos_checkpoints/crosslingual/results'
    os.makedirs(CKPT_DIR, exist_ok=True)
    os.makedirs(RESULTS_DIR, exist_ok=True)
    COLAB = True
    print(f"Checkpoints: {CKPT_DIR}")
    print(f"Results: {RESULTS_DIR}")
except:
    COLAB = False
    CKPT_DIR = './'
    RESULTS_DIR = './results'
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print("Not in Colab. Saving locally.")

## Configuration

**Change `LANGUAGE` and `MODEL_TYPE` below, then Run All.**

Run the notebook 8 times total (4 languages x 2 models) to get all results.
The final cell aggregates everything from saved JSON files.

In [ ]:
# ============================================================
# CONFIGURATION — CHANGE THESE TWO VALUES EACH RUN
# ============================================================

LANGUAGE = "hindi"        # Options: "hindi", "marathi", "urdu", "turkish"
MODEL_TYPE = "synpos"     # Options: "synpos", "mrlpos"

# ============================================================

LANG_CONFIG = {
    "hindi": {
        "name": "Hindi",
        "treebank": "UD_Hindi-HDTB",
        "prefix": "hi_hdtb-ud",
        "typology": "fusional",
        "script": "devanagari",
    },
    "marathi": {
        "name": "Marathi",
        "treebank": "UD_Marathi-UFAL",
        "prefix": "mr_ufal-ud",
        "typology": "fusional",
        "script": "devanagari",
    },
    "urdu": {
        "name": "Urdu",
        "treebank": "UD_Urdu-UDTB",
        "prefix": "ur_udtb-ud",
        "typology": "fusional",
        "script": "nastaliq",
    },
    "turkish": {
        "name": "Turkish",
        "treebank": "UD_Turkish-IMST",
        "prefix": "tr_imst-ud",
        "typology": "agglutinative",
        "script": "latin",
    },
}

cfg = LANG_CONFIG[LANGUAGE]
print(f"Language:  {cfg['name']} ({cfg['typology']})")
print(f"Treebank:  {cfg['treebank']}")
print(f"Model:     {MODEL_TYPE.upper()}")
print(f"Script:    {cfg['script']}")

## 1. Load Universal Dependencies Treebank

In [ ]:
# Cell 5: Download and load any UD treebank
import conllu

def download_ud(treebank, prefix):
    """Download a UD treebank from GitHub."""
    base_url = f"https://raw.githubusercontent.com/UniversalDependencies/{treebank}/master"
    splits = {
        "train": f"{prefix}-train.conllu",
        "dev":   f"{prefix}-dev.conllu",
        "test":  f"{prefix}-test.conllu",
    }

    data_dir = f"ud_{LANGUAGE}"
    os.makedirs(data_dir, exist_ok=True)

    for split, fname in splits.items():
        path = f"{data_dir}/{fname}"
        if not os.path.exists(path):
            print(f"Downloading {split}...")
            !wget -q {base_url}/{fname} -O {path}
            if os.path.getsize(path) == 0:
                print(f"  WARNING: {split} file is empty or download failed")
    return splits, data_dir


def load_conllu(path):
    """Parse CoNLL-U file into (words, tags) pairs."""
    data = []
    with open(path, 'r') as f:
        for sent in conllu.parse(f.read()):
            words = [tok["form"] for tok in sent if isinstance(tok["id"], int)]
            tags = [tok["upos"] for tok in sent if isinstance(tok["id"], int)]
            if words:
                data.append((words, tags))
    return data


splits, data_dir = download_ud(cfg["treebank"], cfg["prefix"])
train_data = load_conllu(f"{data_dir}/{splits['train']}")
dev_data   = load_conllu(f"{data_dir}/{splits['dev']}")
test_data  = load_conllu(f"{data_dir}/{splits['test']}")

# Build tag set
all_tags = set()
for _, tags in train_data + dev_data + test_data:
    all_tags.update(tags)
TAG2IDX = {"PAD": 0}
for i, t in enumerate(sorted(all_tags), 1):
    TAG2IDX[t] = i
IDX2TAG = {v: k for k, v in TAG2IDX.items()}

print(f"\n{cfg['name']} ({cfg['typology']})")
print(f"  Train: {len(train_data)} sentences")
print(f"  Dev:   {len(dev_data)} sentences")
print(f"  Test:  {len(test_data)} sentences")
print(f"  Tags ({len(TAG2IDX)}): {sorted(all_tags)}")
print(f"  Sample: {train_data[0][0][:6]}")
print(f"  Tags:   {train_data[0][1][:6]}")

## 2. Feature Branches

**MRL-POS branch:** Character/akshara n-gram affix extraction + BiLSTM + attention.
Uses akshara splitting for Devanagari scripts, character splitting for others.

**SynPOS branch:** Local context CNN over encoder hidden states. No preprocessing needed.

In [ ]:
# Cell 6: Script-aware affix extraction (for MRL-POS only)

def split_aksharas(word):
    """Split Devanagari word into aksharas (orthographic syllables)."""
    CONSONANTS = set(range(0x0915, 0x093A))  # ka to ha
    NUKTA = 0x093C
    HALANT = 0x094D
    aksharas = []
    current = ""
    for ch in word:
        cp = ord(ch)
        if cp in CONSONANTS and current:
            if not current.endswith(chr(HALANT)):
                aksharas.append(current)
                current = ""
        current += ch
    if current:
        aksharas.append(current)
    return aksharas if aksharas else list(word)


def split_units(word, script):
    """Split word into sub-word units based on script type."""
    if script == "devanagari":
        return split_aksharas(word)
    else:
        # Character-level for Latin, Nastaliq, Tamil, etc.
        return list(word)


class AffixExtractor:
    """Language-agnostic affix extractor using script-appropriate units."""

    def __init__(self, script, max_affixes=8, min_n=2, max_n=5, min_freq=2):
        self.script = script
        self.max_affixes = max_affixes
        self.min_n = min_n
        self.max_n = max_n
        self.min_freq = min_freq
        self.ngram2idx = {"<pad>": 0, "<unk>": 1}
        self.idx = 2

    def build_vocab(self, sentences):
        """Build n-gram vocabulary from training sentences."""
        counts = Counter()
        for words in sentences:
            for word in words:
                units = split_units(word, self.script)
                n_units = len(units)
                for n in range(self.min_n, self.max_n + 1):
                    if n <= n_units:
                        prefix = "".join(units[:n])
                        suffix = "".join(units[-n:])
                        counts[prefix] += 1
                        counts[suffix] += 1

        for ngram, freq in counts.items():
            if freq >= self.min_freq and ngram not in self.ngram2idx:
                self.ngram2idx[ngram] = self.idx
                self.idx += 1

        print(f"Affix vocab: {self.idx} entries (min_freq={self.min_freq})")

    def extract(self, word):
        """Extract affix indices for a word."""
        units = split_units(word, self.script)
        n_units = len(units)
        indices = []
        for n in range(self.min_n, self.max_n + 1):
            if n <= n_units:
                prefix = "".join(units[:n])
                suffix = "".join(units[-n:])
                indices.append(self.ngram2idx.get(prefix, 1))
                indices.append(self.ngram2idx.get(suffix, 1))
        return indices[:self.max_affixes]


# Build affix vocab if using MRL-POS
if MODEL_TYPE == "mrlpos":
    affix_extractor = AffixExtractor(script=cfg["script"])
    train_words = [sent for sent, _ in train_data]
    affix_extractor.build_vocab(train_words)
    AFFIX_VOCAB_SIZE = affix_extractor.idx
else:
    affix_extractor = None
    AFFIX_VOCAB_SIZE = 0
    print("SynPOS mode — no affix extraction needed.")

In [ ]:
# Cell 7: Local Context CNN Branch (for SynPOS)

class LocalContextBranch(nn.Module):
    """Multi-scale CNN over encoder hidden states for local syntactic features."""

    def __init__(self, input_dim=768, output_dim=768, kernel_sizes=(1, 2, 3)):
        super().__init__()
        channels_per_kernel = output_dim // len(kernel_sizes)
        self.remainder = output_dim - channels_per_kernel * len(kernel_sizes)

        self.convs = nn.ModuleList()
        for i, ks in enumerate(kernel_sizes):
            out_ch = channels_per_kernel + (1 if i < self.remainder else 0)
            self.convs.append(
                nn.Conv1d(input_dim, out_ch, ks, padding=ks // 2)
            )

        self.norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, encoder_hidden, attention_mask):
        x = encoder_hidden.transpose(1, 2)
        conv_outs = []
        for conv in self.convs:
            c = F.gelu(conv(x))
            c = c[:, :, :encoder_hidden.size(1)]
            conv_outs.append(c)
        h_syn = torch.cat(conv_outs, dim=1).transpose(1, 2)
        h_syn = h_syn * attention_mask.unsqueeze(-1).float()
        return self.dropout(self.norm(h_syn))

print("LocalContextBranch defined.")

In [ ]:
# Cell 8: Shared model components (identical in both models)

class AffixEmbeddingModule(nn.Module):
    """Affix embedding with BiLSTM + intra-affix attention (MRL-POS only)."""
    def __init__(self, affix_vocab_size, affix_dim=128,
                 hidden_dim=128, output_dim=768, max_affixes=8):
        super().__init__()
        self.max_affixes = max_affixes
        self.embedding = nn.Embedding(affix_vocab_size, affix_dim, padding_idx=0)
        self.pos_encoding = nn.Embedding(max_affixes, affix_dim)
        self.bilstm = nn.LSTM(affix_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attn_vector = nn.Linear(hidden_dim * 2, 1, bias=False)
        self.proj = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, affix_ids):
        B, S, M = affix_ids.shape
        flat = affix_ids.view(B * S, M)
        positions = torch.arange(M, device=affix_ids.device).unsqueeze(0).expand(B * S, -1)
        emb = self.embedding(flat) + self.pos_encoding(positions)
        H, _ = self.bilstm(emb)
        scores = self.attn_vector(H).squeeze(-1)
        mask = (flat == 0)
        scores = scores.masked_fill(mask, -1e9)
        beta = F.softmax(scores, dim=-1)
        H_aff = (beta.unsqueeze(-1) * H).sum(dim=1)
        return self.proj(H_aff).view(B, S, -1)


class LayerWiseAttentionPooling(nn.Module):
    def __init__(self, hidden_dim=768, num_layers=12):
        super().__init__()
        self.layer_weights = nn.Linear(hidden_dim, 1)

    def forward(self, all_hidden_states):
        stacked = torch.stack(all_hidden_states[1:], dim=2)
        scores = self.layer_weights(stacked).squeeze(-1)
        alpha = F.softmax(scores, dim=-1)
        return (alpha.unsqueeze(-1) * stacked).sum(dim=2)


class DualGatingCoAttention(nn.Module):
    def __init__(self, hidden_dim=768):
        super().__init__()
        d = hidden_dim
        self.W_a2c = nn.Linear(2 * d, d)
        self.W_c2a = nn.Linear(2 * d, d)
        self.linear_fuse = nn.Linear(2 * d, d)

    def forward(self, H_cw, H_aw):
        concat = torch.cat([H_cw, H_aw], dim=-1)
        g_a2c = torch.sigmoid(self.W_a2c(concat))
        H_cw_gated = g_a2c * H_cw
        g_c2a = torch.sigmoid(self.W_c2a(concat))
        H_aw_gated = g_c2a * H_aw
        fused = torch.cat([H_cw_gated, H_aw_gated], dim=-1)
        return self.linear_fuse(fused)


print("All shared components defined.")

In [ ]:
# Cell 9: Model classes

class MRLPOS(nn.Module):
    """XLM-R + Affix Branch + Dual-Gating Co-Attention + CRF."""
    def __init__(self, affix_vocab_size, num_tags, hidden_dim=768, max_affixes=8):
        super().__init__()
        self.xlmr = XLMRobertaModel.from_pretrained(
            "xlm-roberta-base", output_hidden_states=True)
        self.layer_pooling = LayerWiseAttentionPooling(hidden_dim)
        self.affix_module = AffixEmbeddingModule(
            affix_vocab_size, output_dim=hidden_dim, max_affixes=max_affixes)
        self.co_attention = DualGatingCoAttention(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, input_ids, attention_mask, affix_ids=None, tags=None):
        H_cw = self.layer_pooling(
            self.xlmr(input_ids=input_ids, attention_mask=attention_mask).hidden_states)
        H_aw = self.affix_module(affix_ids)
        H_fused = self.co_attention(H_cw, H_aw)
        emissions = self.classifier(H_fused)
        if tags is not None:
            return -self.crf(emissions, tags, mask=attention_mask.bool())
        return self.crf.decode(emissions, mask=attention_mask.bool())


class SynPOS(nn.Module):
    """XLM-R + Local Context CNN + Dual-Gating Co-Attention + CRF."""
    def __init__(self, num_tags, hidden_dim=768, kernel_sizes=(1, 2, 3)):
        super().__init__()
        self.xlmr = XLMRobertaModel.from_pretrained(
            "xlm-roberta-base", output_hidden_states=True)
        self.layer_pooling = LayerWiseAttentionPooling(hidden_dim)
        self.syn_branch = LocalContextBranch(
            input_dim=hidden_dim, output_dim=hidden_dim, kernel_sizes=kernel_sizes)
        self.co_attention = DualGatingCoAttention(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, input_ids, attention_mask, tags=None):
        H_cw = self.layer_pooling(
            self.xlmr(input_ids=input_ids, attention_mask=attention_mask).hidden_states)
        H_syn = self.syn_branch(H_cw, attention_mask)
        H_fused = self.co_attention(H_cw, H_syn)
        emissions = self.classifier(H_fused)
        if tags is not None:
            return -self.crf(emissions, tags, mask=attention_mask.bool())
        return self.crf.decode(emissions, mask=attention_mask.bool())


print(f"Model classes defined. Using: {MODEL_TYPE.upper()}")

In [ ]:
# Cell 10: Dataset classes

tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

class MRLPOSDataset(Dataset):
    """Dataset with affix_ids for MRL-POS."""
    def __init__(self, data, tokenizer, affix_extractor, tag2idx, max_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.extractor = affix_extractor
        self.tag2idx = tag2idx
        self.max_len = max_len
        self.max_affixes = affix_extractor.max_affixes

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        words, tags = self.data[idx]
        encoding = self.tokenizer(
            words, is_split_into_words=True,
            padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt")
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        word_ids = encoding.word_ids()

        aligned_tags, aligned_affixes = [], []
        prev_word_id = None
        for i, wid in enumerate(word_ids):
            if wid is None:
                aligned_tags.append(self.tag2idx["PAD"])
                aligned_affixes.append([0] * self.max_affixes)
            elif wid != prev_word_id:
                tag = tags[wid] if tags[wid] in self.tag2idx else "X"
                aligned_tags.append(self.tag2idx.get(tag, self.tag2idx.get("X", 0)))
                affix_ids = self.extractor.extract(words[wid])
                affix_ids = affix_ids + [0] * (self.max_affixes - len(affix_ids))
                aligned_affixes.append(affix_ids[:self.max_affixes])
            else:
                aligned_tags.append(self.tag2idx["PAD"])
                aligned_affixes.append([0] * self.max_affixes)
            prev_word_id = wid

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "affix_ids": torch.tensor(aligned_affixes, dtype=torch.long),
            "tags": torch.tensor(aligned_tags, dtype=torch.long),
        }


class SynPOSDataset(Dataset):
    """Dataset without affix_ids for SynPOS."""
    def __init__(self, data, tokenizer, tag2idx, max_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.tag2idx = tag2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        words, tags = self.data[idx]
        encoding = self.tokenizer(
            words, is_split_into_words=True,
            padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt")
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        word_ids = encoding.word_ids()

        aligned_tags = []
        prev_word_id = None
        for i, wid in enumerate(word_ids):
            if wid is None:
                aligned_tags.append(self.tag2idx["PAD"])
            elif wid != prev_word_id:
                tag = tags[wid] if tags[wid] in self.tag2idx else "X"
                aligned_tags.append(self.tag2idx.get(tag, self.tag2idx.get("X", 0)))
            else:
                aligned_tags.append(self.tag2idx["PAD"])
            prev_word_id = wid

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "tags": torch.tensor(aligned_tags, dtype=torch.long),
        }


print("Dataset classes defined.")

## 3. Training

Same hyperparameters for all languages and both models.
Checkpoint name encodes language + model type so results don't collide.

In [ ]:
# Cell 11: Create datasets, model, and train

# --- Hyperparameters (identical for all runs) ---
BATCH_SIZE = 64
MAX_EPOCHS = 30
PATIENCE = 7
ENCODER_LR = 1e-5
HEAD_LR = 1e-3
WARMUP_RATIO = 0.15
MAX_LEN = 128

# --- Create datasets ---
if MODEL_TYPE == "mrlpos":
    train_ds = MRLPOSDataset(train_data, tokenizer, affix_extractor, TAG2IDX, MAX_LEN)
    dev_ds   = MRLPOSDataset(dev_data, tokenizer, affix_extractor, TAG2IDX, MAX_LEN)
else:
    train_ds = SynPOSDataset(train_data, tokenizer, TAG2IDX, MAX_LEN)
    dev_ds   = SynPOSDataset(dev_data, tokenizer, TAG2IDX, MAX_LEN)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
dev_dl   = DataLoader(dev_ds, batch_size=BATCH_SIZE)

# --- Create model ---
if MODEL_TYPE == "mrlpos":
    model = MRLPOS(
        affix_vocab_size=AFFIX_VOCAB_SIZE,
        num_tags=len(TAG2IDX),
    ).to(device)
else:
    model = SynPOS(
        num_tags=len(TAG2IDX),
    ).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# --- Optimizer ---
optimizer = torch.optim.AdamW([
    {"params": list(model.xlmr.parameters()), "lr": ENCODER_LR},
    {"params": [p for n, p in model.named_parameters()
                if not n.startswith("xlmr.")], "lr": HEAD_LR},
])

total_steps = len(train_dl) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f"Training: {LANGUAGE} / {MODEL_TYPE.upper()}")
print(f"Steps: {total_steps}, Warmup: {warmup_steps}")
print(f"Encoder LR: {ENCODER_LR}, Head LR: {HEAD_LR}")


# --- Evaluation ---
def evaluate(model, dataloader):
    model.eval()
    all_preds, all_golds = [], []
    with torch.no_grad():
        for batch in dataloader:
            if MODEL_TYPE == "mrlpos":
                preds = model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["affix_ids"].to(device),
                )
            else:
                preds = model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                )
            for b in range(len(preds)):
                for pred_idx, gold, m in zip(
                    preds[b], batch["tags"][b], batch["attention_mask"][b]
                ):
                    if m.item() == 1 and gold.item() != 0:
                        all_preds.append(pred_idx)
                        all_golds.append(gold.item())
    acc = accuracy_score(all_golds, all_preds)
    f1 = sklearn_f1(all_golds, all_preds, average="macro", zero_division=0)
    return acc, f1, all_preds, all_golds


# --- Checkpoint ---
ckpt_name = f"{LANGUAGE}_{MODEL_TYPE}_best.pt"
ckpt_path = os.path.join(CKPT_DIR, ckpt_name)
start_epoch = 0
best_f1 = 0
best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

if os.path.exists(ckpt_path):
    print(f"\nLoading checkpoint: {ckpt_name}")
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt.get('epoch', 0)
    best_f1 = ckpt.get('best_f1', 0)
    best_state = ckpt['model_state_dict']
    for _ in range(start_epoch * len(train_dl)):
        scheduler.step()
    print(f"  Resumed: epoch {start_epoch}, F1={best_f1:.4f}")

# --- Training loop ---
train_log = []
no_improve = 0
t0 = time.time()

for epoch in range(start_epoch, MAX_EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_dl:
        optimizer.zero_grad()
        if MODEL_TYPE == "mrlpos":
            loss = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["affix_ids"].to(device),
                tags=batch["tags"].to(device),
            )
        else:
            loss = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                tags=batch["tags"].to(device),
            )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_dl)
    acc, f1, _, _ = evaluate(model, dev_dl)
    lr = scheduler.get_last_lr()[0]
    elapsed = time.time() - t0
    print(f"Epoch {epoch+1:2d} | Loss: {avg_loss:.4f} | "
          f"Acc: {acc:.4f} | F1: {f1:.4f} | LR: {lr:.2e} | {elapsed:.0f}s")

    train_log.append({"epoch": epoch+1, "loss": avg_loss, "acc": acc, "f1": f1})

    if f1 > best_f1:
        best_f1 = f1
        no_improve = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': best_state,
            'optimizer_state_dict': optimizer.state_dict(),
            'best_f1': best_f1,
        }, ckpt_path)
        print(f"  -> Saved ({ckpt_name}, F1={best_f1:.4f})")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}. Best F1: {best_f1:.4f}")
            break

total_time = time.time() - t0
print(f"\nTotal training time: {total_time:.0f}s ({total_time/60:.1f}m)")

In [ ]:
# Cell 12: Final evaluation and save results

print("=" * 60)
print(f"Final report: {cfg['name']} / {MODEL_TYPE.upper()}")
print("=" * 60)

model.load_state_dict(best_state)
model.to(device)
acc, f1, all_preds, all_golds = evaluate(model, dev_dl)

# Classification report
real_tags = sorted([t for t in TAG2IDX.keys() if t != "PAD"])
gold_labels = [IDX2TAG[g] for g in all_golds]
pred_labels = [IDX2TAG[p] for p in all_preds]

# Only include tags that appear in dev set
dev_tags = sorted(set(gold_labels))
print(f"\nDev tags ({len(dev_tags)}): {dev_tags}")
print(classification_report(gold_labels, pred_labels, labels=dev_tags, zero_division=0))

# Per-tag F1
from sklearn.metrics import f1_score
per_tag_f1 = {}
for tag in dev_tags:
    binary_gold = [1 if g == tag else 0 for g in gold_labels]
    binary_pred = [1 if p == tag else 0 for p in pred_labels]
    per_tag_f1[tag] = round(f1_score(binary_gold, binary_pred, zero_division=0), 4)

# Macro-F1 excluding zero-support tags
macro_f1_clean = np.mean(list(per_tag_f1.values()))
print(f"Macro-F1 (dev-tags only): {macro_f1_clean:.4f}")

# Save results to JSON
result = {
    "language": LANGUAGE,
    "language_name": cfg["name"],
    "typology": cfg["typology"],
    "model_type": MODEL_TYPE,
    "best_f1": round(best_f1, 4),
    "macro_f1_clean": round(macro_f1_clean, 4),
    "accuracy": round(acc, 4),
    "per_tag_f1": per_tag_f1,
    "train_size": len(train_data),
    "dev_size": len(dev_data),
    "num_tags": len(TAG2IDX),
    "total_params": total_params,
    "training_log": train_log,
}

result_path = os.path.join(RESULTS_DIR, f"{LANGUAGE}_{MODEL_TYPE}.json")
with open(result_path, 'w') as f:
    json.dump(result, f, indent=2)
print(f"\nResults saved: {result_path}")

## 4. Cross-Lingual Comparison

Run this cell after completing all 8 runs (4 languages x 2 models).
It loads saved JSON results and builds the comparison table.

In [ ]:
# Cell 13: Aggregate results from all runs

import glob

result_files = sorted(glob.glob(os.path.join(RESULTS_DIR, "*.json")))
print(f"Found {len(result_files)} result files:\n")

results = []
for rf in result_files:
    with open(rf) as f:
        results.append(json.load(f))
    print(f"  {os.path.basename(rf)}")

if len(results) < 2:
    print("\nNeed at least 2 results to compare. Run more configurations first.")
else:
    # Build comparison table
    print("\n" + "=" * 80)
    print("CROSS-LINGUAL COMPARISON: SynPOS vs MRL-POS")
    print("=" * 80)
    print(f"{'Language':<12} {'Typology':<15} {'Train':<8} {'MRL-POS F1':<12} {'SynPOS F1':<12} {'Winner':<10}")
    print("-" * 80)

    # Group by language
    by_lang = {}
    for r in results:
        lang = r["language"]
        if lang not in by_lang:
            by_lang[lang] = {}
        by_lang[lang][r["model_type"]] = r

    for lang in ["hindi", "marathi", "urdu", "turkish"]:
        if lang not in by_lang:
            continue
        lr = by_lang[lang]
        name = lr[list(lr.keys())[0]]["language_name"]
        typology = lr[list(lr.keys())[0]]["typology"]
        train_sz = lr[list(lr.keys())[0]]["train_size"]

        mrl_f1 = lr.get("mrlpos", {}).get("macro_f1_clean", None)
        syn_f1 = lr.get("synpos", {}).get("macro_f1_clean", None)

        mrl_str = f"{mrl_f1:.4f}" if mrl_f1 else "—"
        syn_str = f"{syn_f1:.4f}" if syn_f1 else "—"

        if mrl_f1 and syn_f1:
            diff = syn_f1 - mrl_f1
            if abs(diff) < 0.005:
                winner = "TIE"
            elif diff > 0:
                winner = f"SynPOS +{diff:.3f}"
            else:
                winner = f"MRL-POS +{-diff:.3f}"
        else:
            winner = "pending"

        print(f"{name:<12} {typology:<15} {train_sz:<8} {mrl_str:<12} {syn_str:<12} {winner:<10}")

    print("-" * 80)

    # Check hypothesis
    print("\nHypothesis check:")
    for lang, lr in by_lang.items():
        if "mrlpos" in lr and "synpos" in lr:
            typology = lr["mrlpos"]["typology"]
            mrl = lr["mrlpos"]["macro_f1_clean"]
            syn = lr["synpos"]["macro_f1_clean"]
            expected = "synpos" if typology == "fusional" else "mrlpos"
            actual = "synpos" if syn > mrl else "mrlpos"
            match = "CONFIRMED" if expected == actual else "REJECTED"
            print(f"  {lr['mrlpos']['language_name']:>10} ({typology}): "
                  f"expected {expected.upper()}, got {actual.upper()} -> {match}")

In [ ]:
# Cell 14: Visualization

import matplotlib.pyplot as plt

if len(results) >= 4:
    # Grouped bar chart: MRL-POS vs SynPOS per language
    languages = []
    mrl_scores = []
    syn_scores = []
    colors_mrl = []
    colors_syn = []

    for lang in ["hindi", "marathi", "urdu", "turkish"]:
        if lang in by_lang and "mrlpos" in by_lang[lang] and "synpos" in by_lang[lang]:
            lr = by_lang[lang]
            languages.append(lr["mrlpos"]["language_name"])
            mrl_scores.append(lr["mrlpos"]["macro_f1_clean"])
            syn_scores.append(lr["synpos"]["macro_f1_clean"])

    if languages:
        fig, ax = plt.subplots(figsize=(10, 6))
        x = np.arange(len(languages))
        width = 0.35

        bars1 = ax.bar(x - width/2, mrl_scores, width,
                       label='MRL-POS (morphological)', color='coral', edgecolor='black')
        bars2 = ax.bar(x + width/2, syn_scores, width,
                       label='SynPOS (syntactic)', color='steelblue', edgecolor='black')

        ax.set_ylabel('Macro F1', fontsize=13)
        ax.set_title('Cross-Lingual POS Tagging: Morphological vs Syntactic Inductive Bias',
                      fontsize=14)
        ax.set_xticks(x)
        ax.set_xticklabels([f"{l}\n({by_lang[l.lower()]['mrlpos']['typology']})"
                            for l in languages], fontsize=12)
        ax.legend(fontsize=12)
        ax.set_ylim(min(mrl_scores + syn_scores) - 0.02, 1.0)

        # Add value labels
        for bar in bars1 + bars2:
            h = bar.get_height()
            ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                       xytext=(0, 3), textcoords="offset points",
                       ha='center', va='bottom', fontsize=10)

        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, "crosslingual_comparison.png"), dpi=150)
        plt.show()
        print(f"Chart saved to {RESULTS_DIR}/crosslingual_comparison.png")
else:
    print(f"Need results from all 8 runs. Currently have {len(results)}.")
    print("Re-run this cell after completing all configurations.")

## 5. Run Checklist

| # | Language | Model | Config to set | Status |
|---|----------|-------|---------------|--------|
| 1 | Hindi | SynPOS | `LANGUAGE="hindi"`, `MODEL_TYPE="synpos"` | |
| 2 | Hindi | MRL-POS | `LANGUAGE="hindi"`, `MODEL_TYPE="mrlpos"` | |
| 3 | Marathi | SynPOS | `LANGUAGE="marathi"`, `MODEL_TYPE="synpos"` | |
| 4 | Marathi | MRL-POS | `LANGUAGE="marathi"`, `MODEL_TYPE="mrlpos"` | |
| 5 | Urdu | SynPOS | `LANGUAGE="urdu"`, `MODEL_TYPE="synpos"` | |
| 6 | Urdu | MRL-POS | `LANGUAGE="urdu"`, `MODEL_TYPE="mrlpos"` | |
| 7 | Turkish | SynPOS | `LANGUAGE="turkish"`, `MODEL_TYPE="synpos"` | |
| 8 | Turkish | MRL-POS | `LANGUAGE="turkish"`, `MODEL_TYPE="mrlpos"` | |

After all 8 runs, execute Cell 13 and Cell 14 for the comparison table and chart.